# `integrate_results.csv`

Connor Shrader / 09-21-2025

This file loads in and computes metrics on all of the clone results and saves to the file `all_clone_results.csv`.

In [24]:
import os
import pandas as pd
import numpy as np
import ast
import testis_model as tm
from tqdm import tqdm

from saving_loading import find_scenarios, load_all_scenarios, load_clone_names
from testis_model.metrics import init_time, last_time, min_time, max_time, freq_at, size, avg_freq_change, run_likelihood

In [25]:
scenarios = find_scenarios()
fish_data, clone_data = load_all_scenarios(scenarios)
# clone_names = load_clone_names()

In [26]:
print(f'Found scenarios are: {scenarios}.')

Found scenarios are: ['n1000_r10_y6', 'n500_r15_y6', 'n500_r1_y6', 'n1500_r36_y6', 'n500_r15_y9', 'n1000_r1_y3', 'n1000_r1.5_y6', 'n1000_r9_y6', 'n500_r1_y9', 'n1000_r3_y6_realistic', 'n1500_r36_y9', 'n1000_r15_y6', 'n1000_r0_y3', 'n1000_r0.5_y6', 'n1000_r8_y6', 'n500_r4_y6', 'n500_r0_y9', 'n1000_r4_y3', 'n1000_r15_y9', 'no_collapse', 'n1000_r5_y6_realistic', 'n1000_r2_y6_realistic', 'n500_r0_y6', 'n500_r4_y9', 'n1000_r36_y3', 'n1500_r4_y9', 'n1500_r0_y6', 'n1000_r8_y6_realistic', 'n1000_r7_y6', 'n1500_r0_y9', 'n1500_r4_y6', 'full_pipeline_trunc', 'n1000_r3_y6', 'n500_r36_y3', 'n1500_r1_y9', 'n1000_r9_y6_realistic', 'n1000_r2_y6', 'n1500_r15_y3', 'n1500_r1_y6', 'n1000_r6_y6', 'n1000_r1_y6', 'n1500_r36_y3', 'n500_r1_y3', 'n1000_r1.5_y6_realistic', 'n500_r15_y3', 'n1000_r10_y6_realistic', 'n1000_r5_y6', 'n1000_r1_y9', 'n1000_r0.5_y6_realistic', 'n500_r0_y3', 'n1000_r4_y6', 'n1000_r0_y9', 'n500_r4_y3', 'n1000_r0_y6', 'n1000_r4_y9', 'n1000_r15_y3', 'full_pipeline', 'n1500_r4_y3', 'n1000_r3

In [27]:
long_dfs = []
for scenario in scenarios:
    path = os.path.join('scenarios', scenario, f'{scenario}_long.csv')
    if os.path.isfile(path):
        long_dfs.append(pd.read_csv(path))

In [28]:
long_results = pd.concat(long_dfs).set_index(['scenario', 'fish', 'clone'])
print(f'Long results has length {len(long_results)}.')
long_results.head()

Long results has length 44103.


init_time  init_freq      r_opt      l_max     lower  \
scenario     fish clone                                                         
n1000_r10_y6 0    0            5.0   0.307278   9.502927 -54.590756  3.270517   
                  1            5.0   0.261456   9.091371 -53.211698  3.057740   
                  2            5.0   0.153639  14.125175 -37.272727  5.949853   
                  3            5.0   0.150943  10.296517 -53.669315  3.844471   
                  4            5.0   0.086253   6.230935 -50.982342  1.493676   

                             upper      l_null  
scenario     fish clone                         
n1000_r10_y6 0    0      30.590637 -138.671614  
                  1      28.737771  -78.480982  
                  2      37.629256 -174.208461  
                  3      30.870711  -88.089340  
                  4      21.424083  -61.326150

In [29]:
# Specify order of columns
columns = ['scenario', 'fish', 'clone', 'init_time', 'init_freq', 'last_time',
           'min_time', 'max_time', 'max_freq', 'size', 'avg_freq_change', 'r_opt',
           'l_max', 'lower', 'upper', 'l_null']

rows = []
for (scenario, fish, clone), data in tqdm(clone_data.items()):
    try:
        if (scenario, fish, clone) in long_results.index:
            row = long_results.loc[scenario, fish, clone].to_dict()
        else:
            row = {col: pd.NA for col in columns}
            it = init_time(data)
            if it is None: continue
            else: row['init_time'] = it

            row['init_freq'] = freq_at(data, row['init_time'])
    
        row['scenario'] = scenario
        row['fish'] = fish
        row['clone'] = clone
        
        row['last_time'] = last_time(data)
        row['min_time'] = min_time(data)
        row['max_time'] = max_time(data)
        row['max_freq'] = freq_at(data, row['max_time'])
        row['size'] = size(data)
            
        row['avg_freq_change'] = avg_freq_change(data)
        
        rows.append(row)

    except Exception as e:
        print(f'Error for scenario={scenario}, fish={fish}, clone={clone}')
        raise

100%|████████████████████████████████████| 55514/55514 [01:23<00:00, 666.95it/s]


In [30]:
df = pd.DataFrame(rows, columns = columns)
df.to_csv('all_clone_results.csv', index = False)

In [31]:
num_na = len(df[df['r_opt'].isna()])
num_fail = len(df[df['l_max'] < -1e5])

print(f'{num_na} rows have r_opt == NA, and {num_fail} have l_max < -1e5 (indicating a failure to converge).')

39 rows have r_opt == NA, and 79 have l_max < -1e5 (indicating a failure to converge).


In [39]:
len(df[(df['l_max'] < -1e5) & (df['scenario'] == 'full_pipeline_trunc')])

38